# Task 4 — Valutazione dei classificatori manuali su `training.csv`

> 📄 Documentazione completa e motivazioni: [`docs/task4.md`](../docs/task4.md)

## 0. Setup e dati

Il **punto 4 della traccia** chiede di valutare i due classificatori costruiti a mano nel Task 2
(1R e Naïve Bayes) sul dataset reale `training.csv` (o un suo sottoinsieme), cercando di
**ottimizzarne le prestazioni**. A differenza del Task 2, qui lavoriamo su ~41.000 istanze
**fortemente sbilanciate**, vedremo quindi come i due modelli precedentemente addestrati sul dataset da sole 12 righe si comporteranno.

In [6]:
import pandas as pd
import numpy as np




In [7]:
training = pd.read_csv("../data/processed/training.csv", sep=";")
dataFrame = training.copy() # copia del dataset originale così da non modificarlo

## 0.1 Inizializzazione variabili e metodi 

Nelle 2 celle seguenti riprendiamo la variabile **probabilita** , con alcune modifiche per gestire il valore unknown , e la funzione **predici_naive_bayes**, presenti in **task2_NaiveBayes.ipynb**.

In [8]:
#Tutte le probabilità condizionate calcolate con lo stimatore di Laplace.
#Rispetto al Task 2 'marital' acquista un quarto valore, 'unknown' (assente in manuale.csv ma
#presente in training.csv, 80 righe). 'unknown' è una categoria legittima del Bank Marketing,
#non un dato mancante. Avendo ora k=4 categorie, il denominatore di Laplace per 'marital' passa
#da Nc+3=9 a Nc+k=6+4=10: RICALCOLIAMO tutte le sue probabilità così che continuino a sommare a 1.
#(housing e loan avevano già 'unknown' tra i 3 valori, quindi restano con denominatore 9.)
#N.B.: il riscalamento del solo denominatore di 'marital' agisce in egual misura sulle due classi,
#quindi le predizioni del classificatore NON cambiano; cambia solo la validità della distribuzione.
probabilita = {
    "marital": {
        0: {"divorced": 2/13, "married": 8/13, "single": 1/13, "unknown": 2/13},
        1: {"divorced": 3/13, "married": 4/13, "single": 4/13, "unknown": 1/13},
    },
    "housing": {
        0: {"no": 3/9, "unknown": 1/9, "yes": 5/9},
        1: {"no": 3/9, "unknown": 2/9, "yes": 4/9},
    },
    "loan": {
        0: {"no": 6/9, "unknown": 1/9, "yes": 2/9},
        1: {"no": 6/9, "unknown": 2/9, "yes": 1/9},
    }
}

In [9]:
def predici_naive_bayes(row):
    # Probabilità a priori
    score_0 = 0.5
    score_1 = 0.5

    # marital
    score_0 *= probabilita["marital"][0][row["marital"]] 
    score_1 *= probabilita["marital"][1][row["marital"]]

    # housing
    score_0 *= probabilita["housing"][0][row["housing"]]
    score_1 *= probabilita["housing"][1][row["housing"]]

    # loan
    score_0 *= probabilita["loan"][0][row["loan"]]
    score_1 *= probabilita["loan"][1][row["loan"]]



    # Predizione finale 
    if (score_0 > score_1):
        prediction = 0
    else:
        prediction = 1
    return prediction, score_0, score_1

## 1. Naïve Bayes su `training.csv`

Valutiamo il Naïve Bayes (qui nella variante `GaussianNB` di scikit-learn, come controprova
sull'intero dataset). Un punto metodologico importante riguarda il **leakage**: la codifica dei
nominali con `OrdinalEncoder` viene **adattata (`fit`) solo sul training** e poi semplicemente
**applicata (`transform`)** al test, così che nessuna informazione del test influenzi
l'addestramento. Confrontiamo due scenari — **con** e **senza** l'attributo `duration` — per
misurare quanto questa variabile "drogata" dal leakage gonfi i risultati.

In [ ]:
# Usa SOLO le colonne del modello Naive Bayes
training_nb = dataFrame[["marital", "housing", "loan"]].copy()

training_nb["Predicted"] = training_nb.apply(
    lambda row: predici_naive_bayes(row)[0],
    axis=1
)

#chiamata alla funzione predici_naive_bayes , con le probabilità calcolate in precedenza
dataFrame["Predicted"] = dataFrame.apply(lambda row: predici_naive_bayes(row)[0],axis=1)
dataFrame.head(10)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,Predicted
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
5,45,services,married,basic.9y,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
6,59,admin.,married,professional.course,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
7,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
8,24,technician,single,professional.course,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1
9,25,services,single,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1
